# Exploring the HAL/ALMANACH Documents Parquet File

Quick exploration of `/data/gbg141/Arlequin/data/almanach_documents_df.parquet`

In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet('/data/gbg141/Arlequin/data/almanach_documents_df.parquet')
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Dtypes:\n{df.dtypes}")

Shape: (291, 2)
Columns: ['body_text', 'embedding']
Dtypes:
body_text    object
embedding    object
dtype: object


In [3]:
# Preview body_text (truncated)
df_preview = df.copy()
df_preview['body_text_preview'] = df['body_text']
df_preview['text_length'] = df['body_text'].str.len()
df_preview['embedding_dim'] = df['embedding'].apply(len)
df_preview[['body_text_preview', 'text_length', 'embedding_dim']]

,body_text_preview,text_length,embedding_dim
0,The Diccionario da Lingua Portugueza by Antóni...,22311,1024
1,The success of Service Oriented Architecture (...,28410,1024
2,Les modèles de langue contextuels CAMEMBERT po...,20368,1024
3,Advances in natural language processing (NLP) ...,18706,1024
4,Reconnaître les entités mentionnées dans un te...,30188,1024
...,...,...,...
286,Dependency treebanks have become the standard ...,30427,1024
287,Ce rapport présente un état de l'art des métri...,138153,1024
288,L'utilisation des données du rapport de fouill...,487429,1024
289,With its writing system being a Scriptua Conti...,22659,1024


In [4]:
df['body_text'][0]

"The Diccionario da Lingua Portugueza by António de Morais Silva, hereafter referred to as Morais, constitutes a considerable piece of cultural heritage since it marks the beginning of modern Portuguese lexicography, serving also as a model for all subsequent lexicographic production throughout the 19th and 20th centuries. In this paper, we present MORDigital, a newly funded Portuguese lexicographic project, which was successfully submitted to the IC&DT 2020 Projects Call under the scientific area of 'information sciences computing', which falls under 'languages and literatures -linguistics, subarea computer sciences and information sciences'. The project will be funded over the next three years (2021)(2022)(2023)(2024).\nThe MORDigital project aims to produce high-quality and searchable digital versions of the first three editions (1789; 1813; 1823) of Morais in order to preserve this important European heritage work while also making it accessible. These digital versions will be conv

## Text Length Distribution

In [ ]:
import matplotlib.pyplot as plt

text_lengths = df['body_text'].str.len()
print(f"Text length stats:")
print(f"  Min:    {text_lengths.min():,} chars")
print(f"  Max:    {text_lengths.max():,} chars")
print(f"  Mean:   {text_lengths.mean():,.0f} chars")
print(f"  Median: {text_lengths.median():,.0f} chars")
print(f"  Std:    {text_lengths.std():,.0f} chars")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(text_lengths, bins=40, edgecolor='black', alpha=0.7)
ax.set_xlabel('Document length (characters)')
ax.set_ylabel('Count')
ax.set_title(f'Distribution of document lengths (n={len(df)})')
ax.axvline(text_lengths.median(), color='red', linestyle='--', label=f'Median: {text_lengths.median():,.0f}')
plt.legend()
plt.tight_layout()
plt.show()

## Embedding Space

In [ ]:
embeddings = np.array(df['embedding'].to_list())
print(f"Embeddings shape: {embeddings.shape}")
print(f"Dtype: {embeddings.dtype}")
print(f"Value range: [{embeddings.min():.4f}, {embeddings.max():.4f}]")
print(f"Mean norm: {np.linalg.norm(embeddings, axis=1).mean():.4f}")
print(f"Std norm:  {np.linalg.norm(embeddings, axis=1).std():.4f}")

In [ ]:
# 2D projection with t-SNE
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
embeddings_2d = tsne.fit_transform(embeddings)

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], s=20, alpha=0.6)
ax.set_title('t-SNE projection of document embeddings')
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
plt.tight_layout()
plt.show()

## Pairwise Cosine Similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

cos_sim = cosine_similarity(embeddings)
np.fill_diagonal(cos_sim, np.nan)

print(f"Pairwise cosine similarity (excluding self):")
print(f"  Min:    {np.nanmin(cos_sim):.4f}")
print(f"  Max:    {np.nanmax(cos_sim):.4f}")
print(f"  Mean:   {np.nanmean(cos_sim):.4f}")
print(f"  Median: {np.nanmedian(cos_sim):.4f}")

fig, ax = plt.subplots(figsize=(10, 4))
upper_tri = cos_sim[np.triu_indices_from(cos_sim, k=1)]
ax.hist(upper_tri, bins=60, edgecolor='black', alpha=0.7)
ax.set_xlabel('Cosine similarity')
ax.set_ylabel('Count')
ax.set_title('Distribution of pairwise cosine similarities')
plt.tight_layout()
plt.show()

## KMeans Clustering Preview

In [ ]:
from sklearn.cluster import KMeans

norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
normalized = embeddings / np.maximum(norms, 1e-12)

results = {}
for k in [5, 10, 15, 20]:
    km = KMeans(n_clusters=k, n_init=5, random_state=42)
    labels = km.fit_predict(normalized)
    results[k] = labels
    sizes = np.bincount(labels)
    print(f"k={k:2d}: cluster sizes = {sorted(sizes.tolist(), reverse=True)}")

In [ ]:
# Visualize k=10 clusters on t-SNE
labels_10 = results[10]

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(
    embeddings_2d[:, 0], embeddings_2d[:, 1],
    c=labels_10, cmap='tab10', s=25, alpha=0.7
)
ax.set_title('t-SNE colored by spherical KMeans (k=10)')
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
plt.colorbar(scatter, label='Cluster')
plt.tight_layout()
plt.show()

In [ ]:
# Show representative snippet from each cluster (k=10)
for cluster_id in range(10):
    mask = labels_10 == cluster_id
    cluster_docs = df[mask]
    print(f"\n--- Cluster {cluster_id} ({mask.sum()} docs) ---")
    for _, row in cluster_docs.head(3).iterrows():
        print(f"  * {row['body_text'][:150]}...")

## FINCH Hierarchical Clustering Preview

In [ ]:
from hnne.finch_clustering import FINCH

clusters, n_clusters, _, _ = FINCH(
    data=embeddings, distance='cosine', verbose=0, random_state=42
)
print(f"FINCH found {clusters.shape[1]} levels")
print(f"Cluster counts per level: {n_clusters}")
for level in range(clusters.shape[1]):
    sizes = np.bincount(clusters[:, level])
    print(f"  Level {level}: {n_clusters[level]} clusters, sizes = {sorted(sizes.tolist(), reverse=True)}")

In [ ]:
# Visualize first 3 FINCH levels on t-SNE
n_levels = min(3, clusters.shape[1])
fig, axes = plt.subplots(1, n_levels, figsize=(6 * n_levels, 5))
if n_levels == 1:
    axes = [axes]
for level, ax in zip(range(n_levels), axes):
    scatter = ax.scatter(
        embeddings_2d[:, 0], embeddings_2d[:, 1],
        c=clusters[:, level], cmap='tab20', s=20, alpha=0.7
    )
    ax.set_title(f'FINCH Level {level} ({n_clusters[level]} clusters)')
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')
plt.tight_layout()
plt.show()

## Language Detection (quick heuristic)

In [ ]:
french_markers = ['les ', 'des ', 'une ', 'est ', 'dans ', 'pour ', 'nous ', 'cette ']

def guess_language(text):
    first_500 = text[:500].lower()
    fr_count = sum(1 for m in french_markers if m in first_500)
    return 'French' if fr_count >= 3 else 'English'

df['lang'] = df['body_text'].apply(guess_language)
print(df['lang'].value_counts())

# Show on t-SNE
fig, ax = plt.subplots(figsize=(10, 8))
colors = df['lang'].map({'French': 'blue', 'English': 'orange'})
ax.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], c=colors, s=25, alpha=0.7)
ax.set_title('t-SNE colored by detected language')
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
# Manual legend
import matplotlib.patches as mpatches
ax.legend(handles=[
    mpatches.Patch(color='blue', label='French'),
    mpatches.Patch(color='orange', label='English')
])
plt.tight_layout()
plt.show()